# Tutorial 2: Sample Groups and dimensionality-reduction artifacts

This notebook loads the same Metabolic Dataset as tutorial 1, creates Sample Groups from the `cell_type` metadata column, computes PCA, t-SNE, and UMAP, and renders their immutable Analysis Artifacts.


## Before you run it

The notebook is self-contained. Generate the three Th17 example files first; `demo/README.md` lists the command and dependencies.


In [ ]:
# ruff: noqa: E402, I001
import os
import sys
import tempfile
import warnings
from pathlib import Path

from bokeh.io import output_notebook, show

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings(
    "ignore",
    message="n_jobs value 1 overridden to 1 by setting random_state.*",
)


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    AnalysisWorkspace,
    ExperimentAnalysis,
    ExpressionData,
    ExpressionPreprocessingConfig,
    MetabolicModel,
    ReactionData,
    ReactionPreprocessingConfig,
    SampleMetadata,
    build_reduction_presentation,
    plot_reduction_artifact,
)
from metabis.analysis_workspace import RevisionKind
from metabis.version import __version__ as metabis_version

output_notebook()


In [ ]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = repo_root / "data"
required_example_files = (
    "Th17_Compass_reactions.tsv",
    "Th17_Compass_expression.tsv",
    "Th17_Compass_metadata.tsv",
)
missing = [name for name in required_example_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing generated example files: "
        + ", ".join(missing)
        + ". Run `python data/download_and_process_compass.py`."
    )

reaction_data_path = data_dir / "Th17_Compass_reactions.tsv"
expression_data_path = data_dir / "Th17_Compass_expression.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"
target_group_id = "Th17p"
baseline_group_id = "Th17n"

model = MetabolicModel(use_cache=True)
model.apply_alignment_corrections(model.get_alignment_file_path("RECON2"))

reaction_data, reaction_validation = ReactionData.create_with_validation(
    str(reaction_data_path),
    data_mode="dual",
    metabolic_model=model,
    reaction_name_format="rxnRecon3DID",
    csv_separator="\t",
    samples_in="columns",
    check_old_identifiers=True,
    preprocessing=ReactionPreprocessingConfig(compass_activity=True),
)
expression_data, expression_validation = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    gene_format="geneSymbols",
    preprocessing=ExpressionPreprocessingConfig(normalize_total=True),
    data_format="csv",
    csv_separator="\t",
    samples_in="columns",
)
metadata = SampleMetadata.from_table(
    metadata_path,
    sample_id_column="sample",
)

dm = AnalysisWorkspace(
    metabolic_model=model,
    reaction_data=reaction_data,
    expression_data=expression_data,
)
dm.update_sample_metadata(metadata)

print(f"Metabis version: {metabis_version}")
print(f"Loaded {len(reaction_data.get_sample_ids())} aligned samples")


## 1. Create metadata-driven Sample Groups

Sample names are immutable identity, not grouping instructions. The workspace creates atomic Sample Groups from the explicit `cell_type` annotation.


In [ ]:
grouping_result = dm.load_groups_from_metadata(["cell_type"])
group_ids = sorted(dm.get_group_ids())
sample_counts = {
    group_id: len(dm.get_group_samples(group_id)) for group_id in group_ids
}

print(f"Detected groups: {group_ids}")
print(f"Samples per group: {sample_counts}")
print(f"Group colors: {dm.get_group_colors()}")
assert target_group_id in group_ids and baseline_group_id in group_ids
grouping_result


## 2. Compute and install reduction Analysis Artifacts

Calculators return typed results. Each result is projected into an immutable artifact with workspace provenance, then installed through the Analysis Workspace before plotting.


In [ ]:
analysis_dependencies = frozenset(
    {
        RevisionKind.DATASET,
        RevisionKind.METADATA,
        RevisionKind.GROUPS,
        RevisionKind.PREPROCESSING,
    }
)
analysis_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)

analysis = ExperimentAnalysis(dm, auto_run=False)
outcome = analysis.run_analysis(
    include_pca=True,
    pca_data_types=["reactions", "expression"],
    n_components=6,
    n_variable_features=3000,
)
assert outcome["status"] == "success", outcome["messages"]

analysis.calculate_tsne(
    data_type="reactions",
    perplexity=10.0,
    n_variable_features=3000,
)
analysis.calculate_umap(
    data_type="reactions",
    n_neighbors=10,
    min_dist=0.2,
    n_variable_features=3000,
)

for result in analysis.get_reduction_results():
    analysis.install_reduction_artifact(
        result.build_artifact(
            provenance=analysis_snapshot.provenance,
            source_fingerprints=dict(dm.source_fingerprints),
        )
    )

dm.install_experiment_analysis(analysis_snapshot.provenance, analysis)
dm.refresh_experiment_analysis_presentation()
dm.set_analysis_group(target_group_id)

installed = [
    (artifact.key.method, artifact.key.data_type, artifact.source)
    for artifact in analysis.get_reduction_artifacts()
]
print(f"Current mode: {dm.get_mode()}")
print(f"Installed reduction artifacts: {installed}")


## 3. PCA in reaction space

`build_reduction_presentation()` freezes labels, colors, and group membership beside the artifact. `plot_reduction_artifact()` renders only those explicit values.


In [ ]:
pca_reactions_artifact = dm.get_reduction_artifact("PCA", "reactions")
assert pca_reactions_artifact is not None
pca_reactions_artifact_presentation = build_reduction_presentation(
    dm,
    pca_reactions_artifact,
    width=900,
    height=650,
)
show(plot_reduction_artifact(pca_reactions_artifact, pca_reactions_artifact_presentation))


## 4. t-SNE in reaction space

The same plotting entry point works for other reductions once the `ExperimentAnalysis` object has calculated them.

In [ ]:
t_sne_reactions_artifact = dm.get_reduction_artifact("T-SNE", "reactions")
assert t_sne_reactions_artifact is not None
t_sne_reactions_artifact_presentation = build_reduction_presentation(
    dm,
    t_sne_reactions_artifact,
    width=900,
    height=650,
)
show(plot_reduction_artifact(t_sne_reactions_artifact, t_sne_reactions_artifact_presentation))


## 5. UMAP in reaction space

UMAP is often a useful complementary view when you want a more local neighborhood-based embedding of the same samples.

In [ ]:
umap_reactions_artifact = dm.get_reduction_artifact("UMAP", "reactions")
assert umap_reactions_artifact is not None
umap_reactions_artifact_presentation = build_reduction_presentation(
    dm,
    umap_reactions_artifact,
    width=900,
    height=650,
)
show(plot_reduction_artifact(umap_reactions_artifact, umap_reactions_artifact_presentation))


## 6. The same API also works on expression data

The `data_type` argument lets you switch between the reaction matrix and the expression matrix without changing the overall workflow.

In [ ]:
pca_expression_artifact = dm.get_reduction_artifact("PCA", "expression")
assert pca_expression_artifact is not None
pca_expression_artifact_presentation = build_reduction_presentation(
    dm,
    pca_expression_artifact,
    width=900,
    height=650,
)
show(plot_reduction_artifact(pca_expression_artifact, pca_expression_artifact_presentation))


## Takeaways

The workspace now contains metadata-derived Sample Groups and four reduction Analysis Artifacts. Plotting is repeatable because the scientific values and provenance are immutable, while presentation choices are explicit adapters.

The next notebook resolves a Target/Baseline Comparison Membership and installs its comparison artifact.
